# Sidebar Link Validator

This notebook validates all sidebar links and ensures they point to existing pages in the /templates directory. It will automatically create any missing template pages.

In [ ]:
# Import Required Libraries
import os
import json
import shutil
import datetime

# For pretty printing
from pprint import pprint

## Load Sidebar Links

First, we'll load the sidebar links from the configuration file. This could be in JSON, YAML, or another format depending on how your site is configured.

In [ ]:
# Define the path to the sidebar configuration
# This path should be adjusted to match your project structure
sidebar_config_path = "d:/Projects/impressioncore/config/sidebar.json"

# Function to load sidebar links
def load_sidebar_links(config_path):
    try:
        with open(config_path, 'r') as f:
            config = json.load(f)
            return config.get('links', [])
    except FileNotFoundError:
        print(f"Error: Sidebar configuration not found at {config_path}")
        return []
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON in sidebar configuration at {config_path}")
        return []

# Try to load the sidebar links
sidebar_links = load_sidebar_links(sidebar_config_path)

# If the file doesn't exist, we'll create a sample structure for testing
if not sidebar_links:
    print("Using sample sidebar links for testing...")
    sidebar_links = [
        {"title": "Home", "url": "/home"},
        {"title": "About", "url": "/about"},
        {"title": "Services", "url": "/services"},
        {"title": "Blog", "url": "/blog"},
        {"title": "Contact", "url": "/contact"},
        {"title": "FAQ", "url": "/faq"}
    ]

print(f"Loaded {len(sidebar_links)} sidebar links:")
pprint(sidebar_links)

## Check for Missing Pages

Now we'll check if each sidebar link has a corresponding page in the /templates directory.

In [ ]:
# Define the templates directory path
templates_dir = "d:/Projects/impressioncore/templates"

# Ensure the templates directory exists
os.makedirs(templates_dir, exist_ok=True)

# Function to check if a page exists for a given URL
def page_exists(url, templates_dir):
    # Convert URL to a file path (remove leading slash if present)
    if url.startswith('/'):
        url = url[1:]
    
    # Check for both .html and .jinja extensions
    file_path_html = os.path.join(templates_dir, f"{url}.html")
    file_path_jinja = os.path.join(templates_dir, f"{url}.jinja")
    
    return os.path.exists(file_path_html) or os.path.exists(file_path_jinja)

# Check each link and store the results
link_status = []
for link in sidebar_links:
    url = link.get('url', '')
    exists = page_exists(url, templates_dir)
    link_status.append({
        'title': link.get('title', 'Unknown'),
        'url': url,
        'exists': exists
    })

# Display the results
print("\nChecking for missing pages:")
missing_pages = [link for link in link_status if not link['exists']]
existing_pages = [link for link in link_status if link['exists']]

print(f"Found {len(existing_pages)} existing pages:")
for page in existing_pages:
    print(f"✓ {page['title']} ({page['url']})")

print(f"\nFound {len(missing_pages)} missing pages:")
for page in missing_pages:
    print(f"✗ {page['title']} ({page['url']})")

## Create Missing Pages

For each missing page, we'll create a new file with a basic template structure.

In [ ]:
# Basic template for new pages
def create_basic_template(title):
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <link rel="stylesheet" href="/static/css/main.css">
</head>
<body>
    <header>
        <h1>{title}</h1>
    </header>
    
    <main>
        <p>This is the {title} page. Content needs to be added here.</p>
    </main>
    
    <footer>
        <p>&copy; {datetime.datetime.now().year} ImpressionCore. All rights reserved.</p>
    </footer>
</body>
</html>
"""

# Function to create a missing page
def create_missing_page(url, title, templates_dir):
    # Remove leading slash if present
    if url.startswith('/'):
        url = url[1:]
    
    # Create directory structure if needed
    dir_path = os.path.dirname(os.path.join(templates_dir, url))
    os.makedirs(dir_path, exist_ok=True)
    
    # Create file with .html extension
    file_path = os.path.join(templates_dir, f"{url}.html")
    
    with open(file_path, 'w') as f:
        f.write(create_basic_template(title))
    
    return file_path

# Create missing pages
created_pages = []
for page in missing_pages:
    file_path = create_missing_page(page['url'], page['title'], templates_dir)
    created_pages.append({
        'title': page['title'],
        'url': page['url'],
        'file_path': file_path
    })

print(f"\nCreated {len(created_pages)} missing pages:")
for page in created_pages:
    print(f"+ Created: {page['title']} at {page['file_path']}")

## Update Sidebar Links

If needed, we can update the sidebar configuration to ensure all links are correct.

In [ ]:
# This function would update the sidebar configuration if needed
def update_sidebar_config(config_path, updated_links):
    try:
        # Read the existing config
        with open(config_path, 'r') as f:
            config = json.load(f)
        
        # Update the links
        config['links'] = updated_links
        
        # Write back the updated config
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)
            
        return True
    except Exception as e:
        print(f"Error updating sidebar config: {e}")
        return False

# We don't need to update anything in this case, but here's how we could:
# if os.path.exists(sidebar_config_path):
#     update_result = update_sidebar_config(sidebar_config_path, sidebar_links)
#     if update_result:
#         print("Sidebar configuration updated successfully.")
#     else:
#         print("Failed to update sidebar configuration.")
# else:
#     print(f"Sidebar configuration file not found at {sidebar_config_path}. No updates made.")

print("No sidebar configuration updates needed.")

## Generate Report of Link Status

Finally, let's generate a comprehensive report of the link status.

In [ ]:
# Create a summary report
report = {
    "timestamp": datetime.datetime.now().isoformat(),
    "total_links": len(sidebar_links),
    "existing_pages": [{'title': page['title'], 'url': page['url']} for page in existing_pages],
    "created_pages": [{'title': page['title'], 'url': page['url'], 'file_path': page['file_path']} for page in created_pages],
    "all_links_valid": len(missing_pages) == len(created_pages)
}

# Display report
print("\n=== SIDEBAR LINK VALIDATION REPORT ===")
print(f"Timestamp: {report['timestamp']}")
print(f"Total Links: {report['total_links']}")
print(f"Existing Pages: {len(report['existing_pages'])}")
print(f"Created Pages: {len(report['created_pages'])}")
print(f"All Links Valid: {'Yes' if report['all_links_valid'] else 'No'}")

# Save report to file
report_path = "d:/Projects/impressioncore/reports/sidebar_link_validation.json"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nReport saved to: {report_path}")

## Conclusion

The sidebar link validation process is now complete. All missing pages have been created with basic templates.

### Next steps:
1. Review the created pages and add actual content
2. Customize the template structure if needed
3. Set up a regular validation schedule to ensure links remain valid as the site evolves